# EDA - olist_order_reviews_dataset

Exploracao inicial do arquivo de reviews, antes de decidir a estrategia de limpeza.
Perguntas que este notebook responde: quantas reviews tem texto de verdade, qual o tamanho desses textos, como as notas se distribuem, e se ha duplicatas.

In [1]:
import pandas as pd

pd.set_option("display.max_colwidth", 120)

df = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
df.shape

(99224, 7)

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [6]:
df.sample(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
93634,0c0ca022b4342859a09a0e936805d593,d08fd4a6faa8ded70a6af3f2d51eba19,5,NaN,Muito satisfeita recomendo..,2018-04-07 00:00:00,2018-04-08 00:12:58
59128,89fe8a8eecf0297cbd4d46dfb6c435ae,e2638c5bf9a25fab6628686b0ed83f9f,4,NaN,NaN,2017-06-07 00:00:00,2017-06-09 15:08:41
93767,27f3dd0288e889b73cf7089510cc4333,1475f5ec3a992c94a397148561439873,4,NaN,Bom produto,2018-07-14 00:00:00,2018-07-17 22:39:54
28278,21671b095216a3544ae0662bb90f20b9,af86b8dd80134775dceda700fc2ce5ee,5,NaN,"chegou mt rapido, e o produto é excelente, top demais o k10 pro",2017-09-13 00:00:00,2017-09-14 12:31:54
13061,2fd691f0060220072e41b986449df1cb,6fdd6ca7bdcce986b6a442588af840a3,3,NaN,NaN,2018-04-07 00:00:00,2018-04-09 01:54:43
1086,a57e6f022e1156533fb1704f8193d501,a13f2110ae8bab95d529304f3b2dbbc1,1,NaN,NaN,2017-11-19 00:00:00,2017-11-19 16:34:35
68532,e3bcf0222c3f40171365c4ea6ac53051,d6146bd142ef3ffa19fed2bcc5cab997,5,NaN,NaN,2018-05-18 00:00:00,2018-05-21 12:29:19
63440,032f18ccd63ed3488b82e6c36786e93c,a885450c26922878599a0e8ab1bb0569,5,NaN,NaN,2017-12-14 00:00:00,2017-12-15 15:21:40
82121,6acddb0e3f7a754dfc4aaf911d289b6a,ee2fac75aa66d8457b22c46fd08a47f3,5,Otima...,Adorei....,2018-07-21 00:00:00,2018-07-24 05:19:18
29598,4251bcdbfe1c2b461e04df903e3d5436,0396488303af9f0a78feb9ffa7e48949,5,NaN,NaN,2018-08-02 00:00:00,2018-08-04 21:44:50


## Valores ausentes

`review_comment_title` e `review_comment_message` sao os campos de texto livre. Em bases do Olist e comum grande parte das reviews nao ter comentario, so nota.

In [7]:
ausentes = df[["review_comment_title", "review_comment_message"]].isna().sum()
ausentes_pct = (ausentes / len(df) * 100).round(1)
pd.DataFrame({"ausentes": ausentes, "pct": ausentes_pct})

,ausentes,pct
review_comment_title,87656,88.3
review_comment_message,58247,58.7


In [8]:
com_texto = df["review_comment_message"].notna().sum()
print(f"reviews com review_comment_message preenchido: {com_texto} de {len(df)} ({com_texto / len(df):.1%})")

reviews com review_comment_message preenchido: 40977 de 99224 (41.3%)


## Distribuicao das notas (review_score)

In [9]:
df["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

## Tamanho dos comentarios (so quem tem texto)

In [10]:
tamanhos = df["review_comment_message"].dropna().str.len()
tamanhos.describe()

count    40977.000000
mean        68.637675
std         53.849207
min          1.000000
25%         28.000000
50%         53.000000
75%         95.000000
max        208.000000
Name: review_comment_message, dtype: float64

## Duplicatas

In [11]:
print("review_id duplicado:", df["review_id"].duplicated().sum())
print("texto de comentario duplicado (entre quem tem texto):", df["review_comment_message"].dropna().duplicated().sum())

review_id duplicado: 814
texto de comentario duplicado (entre quem tem texto): 4818


## Amostra de comentarios reais

Olhar exemplos de verdade antes de desenhar a funcao de limpeza.

In [12]:
df["review_comment_message"].dropna().sample(10, random_state=42).tolist()

['Um produto como uma carteira somente poderia ser avaliado depois de algum tempo de uso, No entanto, o produto é bem acabado e de couro conforme o especificado pela empresa.',
 'Entrega no prazo , bom produto .',
 'entrega rápida',
 'Chegou no prazo e atendeu as minhas espectativas',
 'Logística ótima entregue antes do prazo previsto, estou satisfeito. ',
 'Entrega no prazo combinado , chegou certinho recomendo esse loja ',
 'Vocês me enviaram faltando 3 cortinas. E agora?',
 'A encomenda demorou muito a chegar mas chegou em perfeito estado.',
 'produto dentro do esperado sem defeitos',
 'Chegou no prazo !']